In [8]:
import os

try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
except Exception:
    from dotenv import load_dotenv
    load_dotenv()

has_openai = bool(os.environ.get("OPENAI_API_KEY"))

if has_openai:
    from langchain_openai import ChatOpenAI, OpenAIEmbeddings
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
    embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
    print("✅ OpenAI 키 OK (질문확장까지 가능)")
else:
    from langchain_huggingface import HuggingFaceEmbeddings
    embeddings = HuggingFaceEmbeddings(
        model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
    )
    print("⚠️ 키 없음 — 필터·MMR은 OK, 질문확장만 건너뜁니다")

from langchain_core.documents import Document
from langchain_chroma import Chroma

✅ OpenAI 키 OK (질문확장까지 가능)


### STEP 1 · 자료 준비 (그대로 실행)
카페 안내문을 청크로 만들고, 각 청크에 category와 source 꼬리표를 붙입니다. 환불 문장을 일부러 비슷하게 여러 개 넣었어요(MMR 실습용).

In [2]:
docs = [
    Document(page_content="에티오피아 예가체프 원두는 꽃향과 산미가 특징입니다.",
             metadata={"category": "원두", "source": "원두안내.pdf"}),
    Document(page_content="브라질 산토스 원두는 고소하고 단맛이 강해 라떼에 잘 어울립니다.",
             metadata={"category": "원두", "source": "원두안내.pdf"}),
    Document(page_content="원두는 개봉 후 2주 안에 드시는 것을 권장합니다.",
             metadata={"category": "원두", "source": "보관법.txt"}),
    Document(page_content="티라미수 케이크는 매일 오전에 소량만 만듭니다.",
             metadata={"category": "디저트", "source": "메뉴판.pdf"}),
    Document(page_content="원두 주문은 결제 후 2~3일 안에 택배로 도착합니다.",
             metadata={"category": "배송", "source": "배송정책.txt"}),
    Document(page_content="환불은 수령 후 7일 이내에 신청할 수 있습니다.",
             metadata={"category": "환불", "source": "환불정책.txt"}),
    Document(page_content="환불 신청은 상품을 받은 날로부터 7일 안에 가능합니다.",
             metadata={"category": "환불", "source": "FAQ.txt"}),
    Document(page_content="수령 후 7일 이내라면 환불 신청이 가능합니다.",
             metadata={"category": "환불", "source": "공지사항.txt"}),
    Document(page_content="환불 금액은 신청 승인 후 영업일 기준 3일 안에 입금됩니다.",
             metadata={"category": "환불", "source": "환불정책.txt"}),
]

vectorstore = Chroma.from_documents(docs, embeddings)

### STEP 2 · 검색기 만들기
벡터DB를 검색기로 바꾸고, 질문과 가장 비슷한 청크 3개를 가져오게 하세요.

In [3]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

# 검색기만 따로 시험해 보기
for d in retriever.invoke("환불은 언제까지 돼요?"):
    print(d.metadata["source"], "|", d.page_content)

환불정책.txt | 환불은 수령 후 7일 이내에 신청할 수 있습니다.
FAQ.txt | 환불 신청은 상품을 받은 날로부터 7일 안에 가능합니다.
환불정책.txt | 환불 금액은 신청 승인 후 영업일 기준 3일 안에 입금됩니다.


### STEP 3 · format_docs 함수
검색된 청크들을 출처를 붙여서 한 글로 합치는 함수를 완성하세요.

In [4]:
def format_docs(docs):
    return "\n\n".join(
        f"[출처: {d.metadata['source']}]\n{d.page_content}"
        for d in docs
    )

### STEP 4 · 가드레일이 있는 프롬프트
자료에 없는 내용은 지어내지 않도록 안전장치 문장을 직접 써 넣으세요.

In [6]:
from langchain_core.prompts import ChatPromptTemplate

rag_prompt = ChatPromptTemplate.from_template("""
당신은 위니브 카페 상담원입니다. 아래 [자료]만 보고 답하세요.
자료에 없는 내용은 '해당 사항은 확인할 수 없습니다'라고 답하세요.
답변 끝에는 출처(출처: 파일명)를 반드시 알려주세요.  # ← 가드레일 문장
답 끝에는 참고한 출처를 적어 주세요.

[자료]
{context}

[질문]
{question}
""")

### STEP 5 · RAG 체인 연결
PART 2에서 채운 빈칸 그대로입니다. 부품을 파이프로 이어 보세요.

In [9]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

rag_chain = (
    {"context": retriever | format_docs,
     "question": RunnablePassthrough()}
    | rag_prompt
    | llm
    | StrOutputParser()
)

print(rag_chain.invoke("환불은 며칠 안에 신청해야 하나요?"))
print(rag_chain.invoke("주차장이 있나요?"))   # 자료에 없는 질문 → 가드레일 확인

환불은 상품을 받은 날로부터 7일 이내에 신청해야 합니다. (출처: 환불정책.txt, FAQ.txt)
해당 사항은 확인할 수 없습니다. (출처: 배송정책.txt)


### STEP 6 · 검색기만 교체하기 (도전)
챗봇 본체는 건드리지 않고 retriever만 바꿔 끼워 봅니다. 체인을 쉽게 다시 만들 수 있도록 함수로 감싸 두면 편해요.

In [10]:
def make_chain(r):
    return ({"context": r | format_docs, "question": RunnablePassthrough()}
            | rag_prompt | llm | StrOutputParser())

# 6-1. 필터: "원두" 분류에서만 찾기
filter_retriever = vectorstore.as_retriever(
    search_kwargs={"k": 3, "filter": {"category": "원두"}}
)

# 6-2. MMR: 후보 6개를 먼저 보고, 그중 서로 다른 3개 고르기
mmr_retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 3, "fetch_k": 6}
)

# 6-3. 질문 확장: 모호한 질문을 AI가 여러 버전으로 바꿔 검색
from langchain_classic.retrievers.multi_query import MultiQueryRetriever
mq_retriever = MultiQueryRetriever.from_llm(retriever=retriever, llm=llm)

print(make_chain(filter_retriever).invoke("추천할 만한 거 있어요?"))
print(make_chain(mmr_retriever).invoke("환불 알려 주세요"))
print(make_chain(mq_retriever).invoke("그거 언제 와요?"))

c:\Users\user\.pyenv\pyenv-win\versions\3.11.9\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


에티오피아 예가체프 원두는 꽃향과 산미가 특징으로, 독특한 맛을 원하신다면 추천드립니다. 또한, 브라질 산토스 원두는 고소하고 단맛이 강해 라떼에 잘 어울리므로 라떼를 좋아하신다면 이 원두도 좋은 선택이 될 것입니다. 

출처: 원두안내.pdf, 보관법.txt
환불은 수령 후 7일 이내에 신청할 수 있으며, 환불 금액은 신청 승인 후 영업일 기준 3일 안에 입금됩니다. 

출처: 환불정책.txt
원두 주문은 결제 후 2~3일 안에 택배로 도착합니다. (출처: 배송정책.txt)
